# Web scraping - Sites dinâmicos

Na aula de hoje veremos:
 - Como encontrar elementos em uma página web
 - Como navegar em diferentes páginas
 - Como preencher formulários
 - Como lidar com estratégias anti-scraping
 - O que são servidores proxies e como utilizá-los
 - Como bloquear requisições

### Encontrando elementos 


| ABORDAGEM       | DESCRIÇÃO                                                       | HTML                                                            | SELENIUM                                                                                                           |
|:----------------|:----------------------------------------------------------------|:----------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------|
| By.ID           | Seleciona elemento HTML com base no id attribute                | \<div id="s-437">...\</div>                                      | find_element(By.ID, "s-437")                                                                                      |
| By.NAME         | Seleciona elemento HTML com base no name attribute              | \<input name="email" />                                         | find_element(By.NAME, "email") <br> find_elements(By.NAME, "email")                                               |
| By.XPATH        | Seleciona elemento HTML que dá match no XPath expression        | \<h1>My <strong>Fantastic</strong> Blog\</h1>                    | find_element(By.XPATH, "//h1/strong") <br> find_elements(By.XPATH, "//h1/strong")                                 |
| By.LINK_TEXT    | Seleciona elemento \<a> HTML que contém o texto do link         | \<a href="/">Home\</a>                                           | find_element(By.LINK_TEXT, "Home") <br> find_elements(By.LINK_TEXT, "Home")                                       |
| By.TAG_NAME     | Seleciona elemento HTML com base no tag name                    | \<span>...\</span>                                               | find_element(By.TAG_NAME, "span") <br> find_elements(By.TAG_NAME, "span")                                         |
| By.CLASS_NAME   | Seleciona elemento HTML com base na class attribute             | \<div class="text-center"><br> Welcome! <br>    \</div>                        | find_element(By.CLASSNAME, "text-center") <br> find_elements(By.CLASSNAME, "text-center")                         |
| By.CSS_SELECTOR | Seleciona elemento HTML que dá match a CSS selector             | \<div class="product-card"> <br>       \<span class="price"\> </br> $140 </br> \</span> <br> \</div>| find_element(By.CSS_SELECTOR, ".product-card .price") <br> find_elements(By.CSS_SELECTOR, ".product-card .price")|


`find_element`: retorna o primeiro elemento que casa com o padrão buscado<br>
`find_elements`: retorna todos os elementos que casam com o padrão buscado     

### Lab

In [2]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait # parecido ao time
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com')
time.sleep(5)
driver.quit()

#### Find elements

In [ ]:
cards = driver.find_elements(By.CLASS_NAME, 'card card_container')
cards = driver.find_elements(By.CSS_SELECTOR, '.card.card_container')
cards = driver.find_elements(By.XPATH, '//*[@id="content-container"]/div/main/div/div[1]')
cards = driver.find_elements(By.TAG_NAME, 'a')

/html/body/main/div/div/main/div/div[1]

#### Find element

In [ ]:
card = driver.find_element(By.CLASS_NAME, 'card card_container')
card = driver.find_element(By.CSS_SELECTOR, '.card.card_container')
card = driver.find_element(By.XPATH, '//*[@id="content-container"]/div/main/div/div[1]')


### Opções

In [ ]:
opts = webdriver.ChromeOptions()

opts.add_argument('--headless=new')
opts.add_argument("--start-maximized")  # abre tela cheia
opts.add_argument("--window-size=1280,800")  # define tamanho manual
opts.add_argument("--incognito")  # modo anônimo
opts.add_argument("--disable-notifications")  # bloqueia pop-ups de notificação
opts.add_argument("--disable-extensions")  # desativa extensões
opts.add_argument("--disable-popup-blocking")  # desativa bloqueador de pop-ups
opts.add_argument("--disable-infobars")  # remove "Chrome is being controlled by automated test software"
opts.add_argument("--no-sandbox")  # útil em servidores Linux
opts.add_argument("--disable-dev-shm-usage")  # previne erros de memória em containers
opts.add_argument("--remote-debugging-port=9222")  # habilita inspeção remota

### Ecommerce (lista de produtos com paginação e preço)

In [4]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())
options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com')

# busca todos os links com o texto 'See Page'
links = driver.find_elements(By.LINK_TEXT, 'See Page')

if links:
    links[0].click()
    print(f'Navegou para: {driver.current_url}')
else:
    print('Nenhum link que atenda ao padrão buscado')

time.sleep(10)

driver.quit()

Navegou para: https://www.scrapingcourse.com/ecommerce/


### Pagination (lista paginada por números)

In [20]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/pagination')

pagina_atual = 1
max_paginas = 3

data = []
while pagina_atual <= max_paginas:
    time.sleep(3)
    items = driver.find_elements(By.TAG_NAME,'a')

    for item in items:
        txt = item.text.strip()
        if '$' in txt:
            name, price = txt.split('$',1)
            data.append({
                'pagina':pagina_atual,
                'nome':name.strip(),
                'preço':price
            })

    prox_pagina = driver.find_elements(By.LINK_TEXT, str(pagina_atual+1))
    if prox_pagina:
        prox_pagina[0].click()
        pagina_atual+=1

print(data)

time.sleep(5)
driver.quit()

[{'pagina': 1, 'nome': 'Chaz Kangeroo Hoodie', 'preço': '52'}, {'pagina': 1, 'nome': 'Teton Pullover Hoodie', 'preço': '70'}, {'pagina': 1, 'nome': 'Bruno Compete Hoodie', 'preço': '63'}, {'pagina': 1, 'nome': 'Frankie Sweatshirt', 'preço': '60'}, {'pagina': 1, 'nome': 'Hollister Backyard Sweatshirt', 'preço': '52'}, {'pagina': 1, 'nome': 'Stark Fundamental Hoodie', 'preço': '42'}, {'pagina': 1, 'nome': 'Hero Hoodie', 'preço': '54'}, {'pagina': 1, 'nome': 'Oslo Trek Hoodie', 'preço': '42'}, {'pagina': 1, 'nome': 'Abominable Hoodie', 'preço': '69'}, {'pagina': 1, 'nome': 'Mach Street Sweatshirt', 'preço': '62'}, {'pagina': 1, 'nome': 'Grayson Crewneck Sweatshirt', 'preço': '64'}, {'pagina': 1, 'nome': 'Ajax Full-Zip Sweatshirt', 'preço': '69'}, {'pagina': 2, 'nome': 'Marco Lightweight Active Hoodie', 'preço': '74'}, {'pagina': 2, 'nome': 'Beaumont Summit Kit', 'preço': '42'}, {'pagina': 2, 'nome': 'Hyperion Elements Jacket', 'preço': '51'}, {'pagina': 2, 'nome': 'Montana Wind Jacket', '

In [21]:
import pandas as pd
df = pd.DataFrame(data)
df

,pagina,nome,preço
0,1,Chaz Kangeroo Hoodie,52
1,1,Teton Pullover Hoodie,70
2,1,Bruno Compete Hoodie,63
3,1,Frankie Sweatshirt,60
4,1,Hollister Backyard Sweatshirt,52
5,1,Stark Fundamental Hoodie,42
6,1,Hero Hoodie,54
7,1,Oslo Trek Hoodie,42
8,1,Abominable Hoodie,69
9,1,Mach Street Sweatshirt,62


### Load More (botão “Load more” para carregar mais itens)

In [20]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementNotInteractableException

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/button-click')

data = []

pagina_atual = 1
for _ in range(100):
    try:
        button = driver.find_element(By.TAG_NAME,'button')
        if button:
            button.click()
            time.sleep(2)
        
            items = driver.find_elements(By.TAG_NAME,'a')
        
            for item in items:
                txt = item.text.strip()
                if '$' in txt:
                    name, price = txt.split('$',1)
                    data.append({
                        'pagina':pagina_atual,
                        'nome':name.strip(),
                        'preço':price
                    })
            pagina_atual += 1
    except ElementNotInteractableException:
        driver.quit()
        break
    
df = pd.DataFrame(data)
df.drop_duplicates(inplace=True)


In [22]:
df

,pagina,nome,preço
0,1,Chaz Kangeroo Hoodie,52
1,1,Teton Pullover Hoodie,70
2,1,Bruno Compete Hoodie,63
3,1,Frankie Sweatshirt,60
4,1,Hollister Backyard Sweatshirt,52
...,...,...,...
1593,15,Leah Yoga Top,39
1594,15,Chloe Compete Tank,39
1595,15,Maya Tunic,29
1596,15,Antonia Racer Tank,34


### Infinite Scrolling (carregar mais itens ao rolar a página)

In [44]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/infinite-scrolling')

for _ in range(20):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

    items = driver.find_elements(By.TAG_NAME,'a')

    for item in items:
        txt = item.text.strip()
        if '$' in txt:
            name, price = txt.split('$',1)
            data.append({
                'pagina':pagina_atual,
                'nome':name.strip(),
                'preço':price
            })
    
df = pd.DataFrame(data)
df.drop_duplicates(inplace=True)

time.sleep(5)
driver.quit()

In [46]:
df

,pagina,nome,preço
0,4,Chaz Kangeroo Hoodie,52
1,4,Teton Pullover Hoodie,70
2,4,Bruno Compete Hoodie,63
3,4,Frankie Sweatshirt,60
4,4,Hollister Backyard Sweatshirt,52
...,...,...,...
1418,4,Leah Yoga Top,39
1419,4,Chloe Compete Tank,39
1420,4,Maya Tunic,29
1421,4,Antonia Racer Tank,34


### Login 

In [9]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
options.add_argument("--disable-notifications")   # bloqueia pop-ups de notificação
options.add_argument("--disable-extensions")      # desativa extensões
options.add_argument("--disable-popup-blocking")  # desativa bloqueador de pop-ups
options.add_argument("--disable-infobars")        # remove "Chrome is being controlled by automated test software"

EMAIL = 'admin@example.com'
PASSWORD = 'password'

driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/login/csrf')

wait = WebDriverWait(driver, 10)

email = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="email"]')))
password = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="password"]')))

email.send_keys(EMAIL)
password.send_keys(PASSWORD)

button = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button')))
button.click()



time.sleep(5)
driver.quit()

### Table Parsing

In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/table-parsing')
time.sleep(5)
tabela = driver.find_element(By.CSS_SELECTOR, 'table')

linhas = tabela.find_elements(By.CSS_SELECTOR, 'tbody tr')
data = []
for linha in linhas:
    cols = [c.text.strip() for c in linha.find_elements(By.TAG_NAME,'td')]
    if cols:
        data.append(cols)

df = pd.DataFrame(data, columns = ['Product ID','Name','Category','Price','In Stock'])

time.sleep(5)
driver.quit()

### Desafios

### Selenium Options

| Opção                                      | Função                                                                   |
| ----------------------------------------------- | ------------------------------------------------------------------------ |
| `--disable-blink-features=AutomationControlled` | Oculta indícios de automação e remove alguns banners visuais.            |
| `--headless=new`                                | Executa sem interface (novo modo headless desde Chrome 109+).            |
| `--no-sandbox`                                  | Necessário em containers/Docker para evitar falhas de segurança.         |
| `--disable-dev-shm-usage`                       | Evita erro de memória compartilhada limitada no Docker.                  |
| `--incognito`                                   | Abre o navegador em modo anônimo (útil para limpar estado).              |
| `--start-maximized`                             | Abre o navegador maximizado (útil para capturas ou cliques precisos).    |
| `--lang=pt-BR`                                  | Define o idioma do navegador (útil para scraping com conteúdo dinâmico). |


#### Opções de proxy e redes

![image](proxy.jpg)


 - https://free-proxy-list.net/pt
 - https://proxydb.net
 - https://proxyscrape.com/free-proxy-list
 - https://hide.mn/en/proxy-list
 - https://pt.proxyscrape.com/free-proxy-list/brazil

### Privacidade e Stealth

### Armadilhas para scrapers (Honeypot Traps)

Uma honeypot trap (ou simplesmente honeypot) é um mecanismo de armadilha anti-scraping. Pode ser um elemento `invisível` ou `disfarçado` em uma página web que usuários humanos normais nunca clicariam nem acessariam, mas que bots automatizados acabam interagindo com — denunciando-se instantaneamente.

#### Links invisíveis

<a href="/ban-user" style="display:none;">Clique aqui</a>

```markdown
<a href="/ban-user" style="display:none;">Clique aqui</a>
```

#### Campos ocultos em formulários

```markdown
<input type="text" name="extra_field" style="display:none;">
```

#### Como fugir de armadilhas de scrapers
 - Evite redes públicas
 - Seja um scraper ético
 - Use headless browsers
 - Evite links ocultos
 - Evite scraper tipo pegue-e-use
 

## Opções do Selenium e Ferramentas de ocultação de automação

| Categoria                       | Option                                          | Descrição                                                | Status Atual                     | Recomendação                                   |
| ------------------------------- | ----------------------------------------------- | -------------------------------------------------------- | -------------------------------- | ---------------------------------------------- |
| **🧱 Interface & UI**           | `--headless=new`                                | Executa o Chrome sem interface (modo invisível).         | ✅ Funcional                      | Use em automações e scraping sem interface.    |
|                                 | `--start-maximized`                             | Abre o navegador maximizado.                             | ✅ Funcional                      | Bom para testes visuais e clicks precisos.     |
|                                 | `--window-size=1920,1080`                       | Define tamanho fixo da janela.                           | ✅ Funcional                      | Útil em servidores ou headless.                |
|                                 | `--disable-infobars`                            | Remove mensagem "controlled by automated test software". | ❌ Obsoleto                       | Sem efeito desde Chrome 76+.                   |
|                                 | `--start-fullscreen`                            | Inicia o navegador em tela cheia.                        | ✅ Funcional                      | Útil em simulações de kiosks ou players.       |
| **🔒 Segurança & Sandbox**      | `--no-sandbox`                                  | Desativa isolamento de processos (necessário em Docker). | ✅ Funcional                      | Use em ambientes containerizados.              |
|                                 | `--disable-dev-shm-usage`                       | Evita falha de memória compartilhada limitada no Docker. | ✅ Funcional                      | Altamente recomendado para containers.         |
|                                 | `--incognito`                                   | Abre o Chrome em modo anônimo.                           | ✅ Funcional                      | Bom para evitar cache e cookies.               |
|                                 | `--disable-gpu`                                 | Desativa aceleração de GPU.                              | ⚠️ Parcialmente funcional        | Só útil em headless antigos.                   |
| **🌐 Rede & Idioma**            | `--lang=pt-BR`                                  | Define idioma do navegador.                              | ✅ Funcional                      | Use para conteúdos localizados.                |
|                                 | `--proxy-server=http://ip:porta`                | Define proxy manual.                                     | ✅ Funcional                      | Útil para rotação de IPs.                      |
|                                 | `--ignore-certificate-errors`                   | Ignora erros SSL/TLS.                                    | ✅ Funcional                      | Use com cautela.                               |
| **🚫 Bloqueios & Notificações** | `--disable-notifications`                       | Bloqueia solicitações de notificações.                   | ✅ Parcialmente funcional         | Bom para evitar pop-ups de permissão.          |
|                                 | `--disable-popup-blocking`                      | Desativa bloqueio de pop-ups.                            | ⚠️ Ignorado em versões recentes. | Use apenas se precisar capturar novas janelas. |
|                                 | `--disable-extensions`                          | Impede o carregamento de extensões.                      | ✅ Funcional                      | Reduz interferências e aumenta velocidade.     |
| **🤖 Antidetecção / Anti-bot**  | `--disable-blink-features=AutomationControlled` | Oculta o fato de ser controlado via automação.           | ✅ Funcional                      | Essencial em sites com proteção anti-bot.      |
|                                 | `user-agent=...`                                | Define manualmente o user agent.                         | ✅ Funcional                      | Pode disfarçar automação.                      |
|                                 | `--profile-directory=Default`                   | Carrega um perfil Chrome existente.                      | ✅ Funcional                      | Para manter sessões autenticadas.              |
|                                 | `--remote-debugging-port=9222`                  | Abre porta para depuração remota.                        | ✅ Funcional                      | Permite monitorar scripts.                     |
| **⚙️ Performance & Logging**    | `--disable-background-timer-throttling`         | Evita pausa de scripts em abas inativas.                 | ✅ Funcional                      | Útil em scraping contínuo.                     |
|                                 | `--disable-backgrounding-occluded-windows`      | Evita “sleep” de janelas fora da tela.                   | ✅ Funcional                      | Evita delays no headless.                      |
|                                 | `--disable-renderer-backgrounding`              | Mantém o renderizador ativo mesmo fora do foco.          | ✅ Funcional                      | Mantém performance estável.                    |
|                                 | `--mute-audio`                                  | Desativa áudio (em players de vídeo).                    | ✅ Funcional                      | Ideal em ambientes headless.                   |
| **📦 Outros úteis**             | `--force-device-scale-factor=1`                 | Corrige zoom em telas 4K.                                | ✅ Funcional                      | Evita cliques imprecisos.                      |
|                                 | `--disable-features=TranslateUI`                | Desativa sugestões de tradução automática.               | ✅ Funcional                      | Evita interferência visual.                    |
|                                 | `--enable-automation`                           | Adiciona tags de automação.                              | ✅ Funcional (ativada por padrão) | Não precisa incluir manualmente.               |


## Ferramentas de Testes

| Ferramenta / Site                                 |                                                       O que testa / fornece | Uso sugerido                                                                       | Observações                                                   |
| ------------------------------------------------- | --------------------------------------------------------------------------: | ---------------------------------------------------------------------------------- | ------------------------------------------------------------- |
| **bot.sannysoft.com**                             |    Conjunto de checks para detectar Selenium/headless e sinais de automação | Verificar quais sinais de automação ainda estão visíveis no seu browser controlado | Muito usado pela comunidade para debug de anti-bot            |
| **Incolumitas – Bot / Headless tests**            |                     Checks variados (web workers, APIs, inconsistências JS) | Testes aprofundados de fingerprint e APIs modernas                                 | Bom para detectar vetores menos óbvios                        |
| **Fingerprint (demo comercial)**                  | Demonstra sinais avançados de fingerprinting usados por soluções comerciais | Avaliar “nível comercial” de detecção em ambiente de teste                         | Ferramenta comercial; demo limitada                           |
| **AmIUnique**                                     |                      Analisa fingerprint do navegador (entropia/uniqueness) | Entender quão único é o seu ambiente e reduzir entropia                            | Útil para planejar mitigação de fingerprint                   |
| **ScrapingCourse (desafios)**                     |                        Páginas didáticas com desafios anti-bot / Cloudflare | Prática com obstáculos reais e educativos                                          | Excelente para treino — já citado por você                    |
| **httpbin.org / httpbin.dev**                     |             Serviço para inspecionar requests (headers, métodos, redirects) | Validar headers, verbos HTTP e comportamento do cliente                            | Ótimo para testes unitários de requests                       |
| **webhook.site**                                  |                     Gera URL temporária para inspecionar payloads recebidos | Testar callbacks, webhooks e ver exatamente o que seu client envia                 | Prático e rápido para debugging de integrações                |
| **BrowserLeaks / BrowserScan**                    |                         Testes de exposição (canvas, fonts, WebGL, plugins) | Checar vetores de fingerprint como canvas/WebGL/fonts                              | Complementa AmIUnique para vetores técnicos                   |
| **Pixelscan**                                     |                     Ferramenta/serviço para analisar fingerprint e trackers | Auditoria de elementos de fingerprint e trackers                                   | Útil em avaliação de privacidade e fingerprint                |


### Testes